In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dropout, Dense, Input

In [ ]:
train_df = pd.read_csv('Google_Stock_Price_Train.csv')
test_df = pd.read_csv('Google_Stock_Price_Test.csv')

In [ ]:
#
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df[['Open']].values)

plt.plot(train_scaled)
plt.title("Scaled Google Stock Prices")
plt.xlabel("Time")
plt.ylabel("Scaled Price")
plt.show()

In [ ]:
#120 previous stock prices are used as input.
#The model will learn to predict the next stock price based on these 120 previous prices.
#
time_step = 120
x_train, y_train = [], []

for i in range(time_step, train_scaled.shape[0]):
    x_train.append(train_scaled[i-time_step:i, 0])
    y_train.append(train_scaled[i, 0])

x_train, y_train = np.array(x_train), np.array(y_train)

In [ ]:
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

In [ ]:
model = Sequential([
    Input(shape=(x_train.shape[1], 1)),
    LSTM(100, activation='tanh', return_sequences=True), Dropout(0.3),
    LSTM(100, activation='tanh'), Dropout(0.3),
    Dense(1)
])

model.compile(optimizer='adam', loss='mae')
model.summary()

In [ ]:
model.fit(x_train, y_train, epochs=200, batch_size=50, validation_split=0.05)

In [ ]:
data = pd.concat((train_df['Open'], test_df['Open']), axis=0)
test_input = data[len(data) - len(test_df) - time_step:].values.reshape(-1, 1)

test_scaled = scaler.transform(test_input)

In [ ]:
x_test = []
for i in range(time_step, test_scaled.shape[0]):
    x_test.append(test_scaled[i-time_step:i, 0])

x_test = np.array(x_test)
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

In [ ]:
y_pred = model.predict(x_test)
y_pred = scaler.inverse_transform(y_pred)

In [ ]:
y_test = test_df['Open'].values
output = model.evaluate(x_test, y_test)

In [ ]:
plt.plot(y_test, color='red', label='Real Stock Price')
plt.plot(y_pred, color='blue', label='Predicted Stock Price')
plt.title('Google Stock Price Prediction')
plt.xlabel('Time')
plt.ylabel('Price')
plt.legend()
plt.show()

In [ ]:
# Google Stock Price Prediction using LSTM – Line-by-Line Comments

```python
# Import Pandas library for data handling and CSV file operations
import pandas as pd

# Import NumPy library for numerical computations and arrays
import numpy as np

# Import Matplotlib library for plotting graphs
import matplotlib.pyplot as plt

# Import MinMaxScaler for scaling data between 0 and 1
from sklearn.preprocessing import MinMaxScaler

# Import Sequential model to build neural network layer-by-layer
from keras.models import Sequential

# Import LSTM, Dropout, Dense, and Input layers for deep learning model
from keras.layers import LSTM, Dropout, Dense, Input
```

```python
# Read training dataset CSV file
train_df = pd.read_csv('Google_Stock_Price_Train.csv')

# Read testing dataset CSV file
test_df = pd.read_csv('Google_Stock_Price_Test.csv')
```

```python
# Create MinMaxScaler object to normalize values between 0 and 1
scaler = MinMaxScaler(feature_range=(0, 1))

# Scale the 'Open' stock price column from training dataset
train_scaled = scaler.fit_transform(train_df[['Open']].values)

# Plot scaled stock prices
plt.plot(train_scaled)

# Set graph title
plt.title("Scaled Google Stock Prices")

# Label x-axis
plt.xlabel("Time")

# Label y-axis
plt.ylabel("Scaled Price")

# Display graph
plt.show()
```

```python
# 120 previous stock prices are used as input
# The model learns to predict next stock price from previous 120 values

# Define number of previous time steps
time_step = 120

# Create empty lists for training input and output
x_train, y_train = [], []

# Loop through training data starting from 120th row
for i in range(time_step, train_scaled.shape[0]):
    
    # Store previous 120 stock prices as input sequence
    x_train.append(train_scaled[i-time_step:i, 0])
    
    # Store current stock price as output value
    y_train.append(train_scaled[i, 0])

# Convert training lists into NumPy arrays
x_train, y_train = np.array(x_train), np.array(y_train)
```

```python
# Reshape x_train into 3D format required for LSTM
# Shape format = (samples, time steps, features)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
```

```python
# Create Sequential LSTM model
model = Sequential([
    
    # Define input layer shape
    Input(shape=(x_train.shape[1], 1)),
    
    # Add first LSTM layer with 100 neurons
    # return_sequences=True passes output to next LSTM layer
    LSTM(100, activation='tanh', return_sequences=True),
    
    # Dropout layer randomly disables 30% neurons to reduce overfitting
    Dropout(0.3),
    
    # Add second LSTM layer with 100 neurons
    LSTM(100, activation='tanh'),
    
    # Apply dropout again
    Dropout(0.3),
    
    # Dense output layer with 1 neuron for stock price prediction
    Dense(1)
])

# Compile model using Adam optimizer and Mean Absolute Error loss
model.compile(optimizer='adam', loss='mae')

# Display model architecture summary
model.summary()
```

```python
# Train the LSTM model using training data
# epochs=200 means complete dataset is trained 200 times
# batch_size=50 means 50 samples processed at one time
# validation_split=0.05 reserves 5% data for validation
model.fit(x_train, y_train, epochs=200, batch_size=50, validation_split=0.05)
```

```python
# Combine training and testing 'Open' price data
# Needed to prepare previous 120 values for test prediction
data = pd.concat((train_df['Open'], test_df['Open']), axis=0)

# Extract required input values for testing
# Includes previous 120 days before test set starts
test_input = data[len(data) - len(test_df) - time_step:].values.reshape(-1, 1)

# Scale test input using same scaler used for training data
test_scaled = scaler.transform(test_input)
```

```python
# Create empty list for test input sequences
x_test = []

# Loop through scaled test data
for i in range(time_step, test_scaled.shape[0]):
    
    # Store previous 120 values as input sequence
    x_test.append(test_scaled[i-time_step:i, 0])

# Convert list into NumPy array
x_test = np.array(x_test)

# Reshape into 3D format for LSTM model
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))
```

```python
# Predict stock prices using trained model
y_pred = model.predict(x_test)

# Convert predicted scaled values back to original stock prices
y_pred = scaler.inverse_transform(y_pred)
```

```python
# Extract actual stock prices from test dataset
y_test = test_df['Open'].values

# Evaluate model performance on test data
output = model.evaluate(x_test, y_test)
```

```python
# Plot actual stock prices in red color
plt.plot(y_test, color='red', label='Real Stock Price')

# Plot predicted stock prices in blue color
plt.plot(y_pred, color='blue', label='Predicted Stock Price')

# Set graph title
plt.title('Google Stock Price Prediction')

# Label x-axis
plt.xlabel('Time')

# Label y-axis
plt.ylabel('Price')

# Display graph legend
plt.legend()

# Show graph
plt.show()
```

# Overall Working of Project

1. Google stock price dataset is loaded using Pandas.
2. Stock prices are normalized using MinMaxScaler.
3. Previous 120 stock prices are used to predict the next value.
4. Training data is reshaped into 3D format for LSTM.
5. An LSTM deep learning model is created.
6. The model is trained using historical stock price data.
7. Test data is prepared using previous time-step sequences.
8. The trained model predicts future stock prices.
9. Predicted values are converted back to original scale.
10. Actual and predicted stock prices are compared using a graph.
